In [1]:
import pandas as pd
import numpy as np
import requests
import time
import mygene
import myvariant

from Bio import Entrez, SeqIO
from Bio.Seq import Seq

In [2]:
df = pd.read_csv("datosGene4PD/t_common_variant.txt", sep = "\t", index_col = False)

In [3]:
df

,Chr,gene_symbol,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_P,joint_phase_OR,joint_phase_OR_CI,pubMed_ID,Unnamed: 10
0,6,GPR126,rs757765789,142758601,T,G,1.42E-06,1.06,1.016-1.098,28256260,NaN
1,1,SYT11,rs202015799,155839054,C,T,4.70E-09,-,-,24842889,NaN
2,12,SLC2A13,rs1994090,40428561,G,T,3.20E-54,12.05,8.35-17.41,24842889,NaN
3,12,SLC2A13,rs2708453,40478652,G,T,3.62E-54,12.05,8.35-17.41,24842889,NaN
4,12,SLC2A13,rs4768212,40474147,C,T,3.62E-54,12.05,8.35-17.41,24842889,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1053,rs117896735,INPP5F,13,U,13,U,1.21E-11,1.77,-,29700661,NaN
1054,rs12456492,RIT2,21,U,21,U,2.15E-11,1.1,-,29700661,NaN
1055,rs7155501,GCH1,15,U,15,U,1.25E-10,1.12,-,29700661,NaN
1056,rs10797576,SIPA1L2,21,U,21,U,1.76E-10,1.13,-,29700661,NaN


In [4]:
df_filtrado = df[df["SNPs_symbol"].str.startswith('rs')].reset_index(drop = True)

In [5]:
df_filtrado = df_filtrado.drop(["Unnamed: 10"], axis = 1)

In [6]:
df_filtrado = df_filtrado.dropna(subset = ['gene_symbol']).reset_index(drop = True)

In [42]:
mv = myvariant.MyVariantInfo()

In [43]:
prueba = mv.querymany(['rs757765789'], scopes = "dbsnp.rsid", fields = "dbsnp", species = "human")

In [47]:
prueba2 = mv.getvariant('rs202015799', fields = "dbsnp,gwascatalog,clinvar")

In [27]:
prueba[0]["dbsnp"]["gene"]["strand"]

'+'

In [7]:
def busca_rsIDs_cadena(df):
    
    lista_rsids = df["SNPs_symbol"].unique().tolist()

    mv = myvariant.MyVariantInfo()

    resultados = mv.querymany(lista_rsids, scopes = "dbsnp.rsid", fields = "dbsnp", species = "human")

    diccionario_snps = {}

    cromosomas_validos = [str(i) for i in range(1, 23)] + ["X", "Y", "MT"]

    bases_validas = ["A", "C", "G", "T", "a", "c", "g", "t"]

    cadenas_validas = ["+", "-", "1", "-1"]

    for resultado in resultados:
        rsid = resultado.get("query")

        if "notfound" in resultado:
            continue

        info_dbsnp = resultado.get("dbsnp", {})

        hg19 = info_dbsnp.get("hg19", {})

        if not hg19:
            continue

        if isinstance(hg19, list):
            hg19 = hg19[0]

        cromosoma = str(info_dbsnp.get("chrom"))
        if cromosoma not in cromosomas_validos:
            continue

        pos_hg19 = hg19.get("start")
    
        ef_allele = info_dbsnp.get("ref", '')
        alt_allele = info_dbsnp.get("alt", '')

        if ef_allele not in bases_validas or alt_allele not in bases_validas:
            continue

        gene_info = info_dbsnp.get("gene", {})
        gene_symbol = "inter"
        cadena = None

        if isinstance(gene_info, dict):
            gene_symbol = gene_info.get("symbol", "inter")
            cadena = gene_info.get("strand", None)

        elif isinstance(gene_info, list):
            gene_symbol = gene_info[0].get("symbol", "inter")
            cadena = gene_info[0].get("strand", None)

        if rsid not in diccionario_snps:

            diccionario_snps[rsid] = {"SNPs_symbol": rsid, "Chr_corr": cromosoma, "SNP_position_corr": pos_hg19, "Effect_allele_corr": ef_allele, "Alternate_allele_corr": [alt_allele], "Gene_symbol_corr": gene_symbol, "Cadena": cadena}

        else:

            if alt_allele not in diccionario_snps[rsid]["Alternate_allele_corr"]:
                diccionario_snps[rsid]["Alternate_allele_corr"].append(alt_allele)

    datos_corregidos = list(diccionario_snps.values())

    return pd.DataFrame(datos_corregidos)

In [8]:
df_corr = busca_rsIDs_cadena(df_filtrado)

474 input query terms found dup hits:	[('rs202015799', 2), ('rs1994090', 3), ('rs2708453', 2), ('rs4768212', 2), ('rs7304281', 2), ('rs204
1 input query terms found no hit:	['rs2740594c']


In [9]:
df_corr

,SNPs_symbol,Chr_corr,SNP_position_corr,Effect_allele_corr,Alternate_allele_corr,Gene_symbol_corr,Cadena
0,rs757765789,6,142758601,T,[G],ADGRG6,+
1,rs202015799,1,155839054,C,"[G, T]",SYT11,+
2,rs1994090,12,40428561,G,"[A, T, C]",SLC2A13,-
3,rs2708453,12,40478652,G,"[A, T]",SLC2A13,-
4,rs4768212,12,40474147,C,"[A, T]",SLC2A13,-
...,...,...,...,...,...,...,...
881,rs666463,17,76425480,A,[T],DNAH17,-
882,rs1941685,18,31304318,G,"[T, C]",ASXL3,+
883,rs8087969,18,48683589,T,"[G, A]",inter,None
884,rs77351827,20,6006041,C,[T],CRLS1,+


In [10]:
df_nuevo = pd.merge(df_filtrado, df_corr, on = "SNPs_symbol", how = "left")

In [11]:
df_nuevo

,Chr,gene_symbol,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_P,joint_phase_OR,joint_phase_OR_CI,pubMed_ID,Chr_corr,SNP_position_corr,Effect_allele_corr,Alternate_allele_corr,Gene_symbol_corr,Cadena
0,6,GPR126,rs757765789,142758601,T,G,1.42E-06,1.06,1.016-1.098,28256260,6,142758601.0,T,[G],ADGRG6,+
1,1,SYT11,rs202015799,155839054,C,T,4.70E-09,-,-,24842889,1,155839054.0,C,"[G, T]",SYT11,+
2,12,SLC2A13,rs1994090,40428561,G,T,3.20E-54,12.05,8.35-17.41,24842889,12,40428561.0,G,"[A, T, C]",SLC2A13,-
3,12,SLC2A13,rs2708453,40478652,G,T,3.62E-54,12.05,8.35-17.41,24842889,12,40478652.0,G,"[A, T]",SLC2A13,-
4,12,SLC2A13,rs4768212,40474147,C,T,3.62E-54,12.05,8.35-17.41,24842889,12,40474147.0,C,"[A, T]",SLC2A13,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1009,17,DNAH17,rs666463,76425480,A,T,2.90E-09,1.08,-,https://doi.org/10.1101/388165,17,76425480.0,A,[T],DNAH17,-
1010,18,ASXL3,rs1941685,31304318,T,G,1.61E-08,1.05,-,https://doi.org/10.1101/388165,18,31304318.0,G,"[T, C]",ASXL3,+
1011,18,MEX3C,rs8087969,48683589,T,G,1.46E-08,0.94,-,https://doi.org/10.1101/388165,18,48683589.0,T,"[G, A]",inter,None
1012,20,CRLS1,rs77351827,6006041,T,C,7.94E-09,1.08,-,https://doi.org/10.1101/388165,20,6006041.0,C,[T],CRLS1,+


In [12]:
df_nuevo = df_nuevo[["Chr", "SNPs_symbol", "SNP_position", "effect_allele", "alternate_allele", "joint_phase_OR", "Chr_corr", "SNP_position_corr", "Effect_allele_corr", "Alternate_allele_corr", "Cadena"]]

In [13]:
df_nuevo

,Chr,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_OR,Chr_corr,SNP_position_corr,Effect_allele_corr,Alternate_allele_corr,Cadena
0,6,rs757765789,142758601,T,G,1.06,6,142758601.0,T,[G],+
1,1,rs202015799,155839054,C,T,-,1,155839054.0,C,"[G, T]",+
2,12,rs1994090,40428561,G,T,12.05,12,40428561.0,G,"[A, T, C]",-
3,12,rs2708453,40478652,G,T,12.05,12,40478652.0,G,"[A, T]",-
4,12,rs4768212,40474147,C,T,12.05,12,40474147.0,C,"[A, T]",-
...,...,...,...,...,...,...,...,...,...,...,...
1009,17,rs666463,76425480,A,T,1.08,17,76425480.0,A,[T],-
1010,18,rs1941685,31304318,T,G,1.05,18,31304318.0,G,"[T, C]",+
1011,18,rs8087969,48683589,T,G,0.94,18,48683589.0,T,"[G, A]",None
1012,20,rs77351827,6006041,T,C,1.08,20,6006041.0,C,[T],+


In [14]:
df_puro = df_nuevo.dropna(subset = ["SNP_position_corr"]).reset_index(drop = True)
df_puro = df_puro.dropna(subset = ["Cadena"]).reset_index(drop = True)

In [15]:
df_puro

,Chr,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_OR,Chr_corr,SNP_position_corr,Effect_allele_corr,Alternate_allele_corr,Cadena
0,6,rs757765789,142758601,T,G,1.06,6,142758601.0,T,[G],+
1,1,rs202015799,155839054,C,T,-,1,155839054.0,C,"[G, T]",+
2,12,rs1994090,40428561,G,T,12.05,12,40428561.0,G,"[A, T, C]",-
3,12,rs2708453,40478652,G,T,12.05,12,40478652.0,G,"[A, T]",-
4,12,rs4768212,40474147,C,T,12.05,12,40474147.0,C,"[A, T]",-
...,...,...,...,...,...,...,...,...,...,...,...
770,17,rs61169879,59917366,T,C,1.09,17,59917366.0,C,"[A, T]",-
771,17,rs666463,76425480,A,T,1.08,17,76425480.0,A,[T],-
772,18,rs1941685,31304318,T,G,1.05,18,31304318.0,G,"[T, C]",+
773,20,rs77351827,6006041,T,C,1.08,20,6006041.0,C,[T],+


In [16]:
df_puro["SNP_position_corr"] = df_puro["SNP_position_corr"].astype(int)

In [17]:
def valida_alelos_SNP(fila):

    effect_allele_original = fila["effect_allele"]
    alternate_allele_original = fila["alternate_allele"]
    effect_allele_rsID = fila["Effect_allele_corr"]
    alternate_allele_rsID = fila["Alternate_allele_corr"]

    alelos_original = [effect_allele_original, alternate_allele_original]
    alelos_rsID = [effect_allele_rsID] + alternate_allele_rsID

    if all(alelo in alelos_rsID for alelo in alelos_original):
        
        return "Coincide perfecto"

    complementario = {"A": "T", "T": "A", "C": "G", "G": "C"}
    alelos_original_complementario = [complementario.get(alelo, alelo) for alelo in alelos_original]

    if all(alelo in alelos_rsID for alelo in alelos_original_complementario):
        
        return "Coincide complementario"

    return "No coincide"

In [18]:
df_puro["Valida_alelos"] = df_puro.apply(valida_alelos_SNP, axis = 1)

In [19]:
print(df_puro["Valida_alelos"].value_counts())

Coincide perfecto          762
No coincide                 10
Coincide complementario      3
Name: Valida_alelos, dtype: int64


In [20]:
df_puro = df_puro[df_puro["Valida_alelos"] != "No coincide"].reset_index(drop = True)

In [21]:
complementario = {"A": "T", "T": "A", "C": "G", "G": "C"}

In [22]:
filas_comp = df_puro["Valida_alelos"] == "Coincide complementario"
df_puro.loc[filas_comp, "effect_allele"] = df_puro.loc[filas_comp, "effect_allele"].map(complementario)
df_puro.loc[filas_comp, "alternate_allele"] = df_puro.loc[filas_comp, "alternate_allele"].map(complementario)

In [23]:
df_puro

,Chr,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_OR,Chr_corr,SNP_position_corr,Effect_allele_corr,Alternate_allele_corr,Cadena,Valida_alelos
0,6,rs757765789,142758601,T,G,1.06,6,142758601,T,[G],+,Coincide perfecto
1,1,rs202015799,155839054,C,T,-,1,155839054,C,"[G, T]",+,Coincide perfecto
2,12,rs1994090,40428561,G,T,12.05,12,40428561,G,"[A, T, C]",-,Coincide perfecto
3,12,rs2708453,40478652,G,T,12.05,12,40478652,G,"[A, T]",-,Coincide perfecto
4,12,rs4768212,40474147,C,T,12.05,12,40474147,C,"[A, T]",-,Coincide perfecto
...,...,...,...,...,...,...,...,...,...,...,...,...
760,17,rs61169879,59917366,T,C,1.09,17,59917366,C,"[A, T]",-,Coincide perfecto
761,17,rs666463,76425480,A,T,1.08,17,76425480,A,[T],-,Coincide perfecto
762,18,rs1941685,31304318,T,G,1.05,18,31304318,G,"[T, C]",+,Coincide perfecto
763,20,rs77351827,6006041,T,C,1.08,20,6006041,C,[T],+,Coincide perfecto


In [24]:
print(df_puro["Cadena"].value_counts(dropna = False))

+    403
-    362
Name: Cadena, dtype: int64


In [25]:
df_puro_OR = df_puro[~df_puro["joint_phase_OR"].isin(["-", "NA", "NaN"])]
df_puro_OR = df_puro_OR.dropna(subset = ["joint_phase_OR"]).reset_index(drop = True)

In [26]:
df_puro_OR

,Chr,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_OR,Chr_corr,SNP_position_corr,Effect_allele_corr,Alternate_allele_corr,Cadena,Valida_alelos
0,6,rs757765789,142758601,T,G,1.06,6,142758601,T,[G],+,Coincide perfecto
1,12,rs1994090,40428561,G,T,12.05,12,40428561,G,"[A, T, C]",-,Coincide perfecto
2,12,rs2708453,40478652,G,T,12.05,12,40478652,G,"[A, T]",-,Coincide perfecto
3,12,rs4768212,40474147,C,T,12.05,12,40474147,C,"[A, T]",-,Coincide perfecto
4,14,rs7304281,40465942,T,C,12.05,12,126945694,T,"[G, C]",+,Coincide perfecto
...,...,...,...,...,...,...,...,...,...,...,...,...
704,17,rs61169879,59917366,T,C,1.09,17,59917366,C,"[A, T]",-,Coincide perfecto
705,17,rs666463,76425480,A,T,1.08,17,76425480,A,[T],-,Coincide perfecto
706,18,rs1941685,31304318,T,G,1.05,18,31304318,G,"[T, C]",+,Coincide perfecto
707,20,rs77351827,6006041,T,C,1.08,20,6006041,C,[T],+,Coincide perfecto


In [27]:
archivo_genoma = "datosGene4PD/Homo_sapiens.GRCh37.completo.fa"

genoma = SeqIO.index(archivo_genoma, "fasta")

In [25]:
def extrae_region_sana(df_final_fila, ventana = 500):

    cromosoma = str(df_final_fila["Chr_corr"])
    pos_snp = df_final_fila["SNP_position_corr"]

    alelo_sano = df_final_fila["effect_allele"]

    secuencia_chr = genoma[cromosoma].seq

    id_snp_python = pos_snp - 1

    inicio = max(0, id_snp_python - ventana)
    fin = id_snp_python + ventana + 1

    molde_forward = str(secuencia_chr[inicio : fin]).upper()

    pos_relativa_snp = id_snp_python - inicio

    reg_sana_forward = molde_forward[:pos_relativa_snp] + alelo_sano + molde_forward[pos_relativa_snp + 1:]
    
    
    cadena = df_final_fila.get("Cadena")
    if cadena == 1 or cadena == "+":
        
        reg_sana = reg_sana_forward

    else:

        reg_sana = str(Seq(reg_sana_forward).reverse_complement())

    return reg_sana.lower()

In [26]:
def extrae_region_parkinson(df_final_fila, reg_sana, ventana = 500):
    
    cadena = df_final_fila.get("Cadena")
    pos_snp = df_final_fila["SNP_position_corr"]

    alelo_parkinson = df_final_fila["alternate_allele"]

    id_snp_python = pos_snp - 1
    inicio = max(0, id_snp_python - ventana)
    pos_relativa_snp = id_snp_python - inicio

    if cadena == 1 or cadena == "+":

        snp = pos_relativa_snp
        nuc = alelo_parkinson
    
    else:

        snp = len(reg_sana) - 1 - pos_relativa_snp
        nuc = str(Seq(alelo_parkinson).complement())

    nuc = nuc.lower()

    reg_park = reg_sana[:snp] + nuc + reg_sana[snp + 1:]

    return reg_park

In [32]:
secuencias = []
etiquetas = []
# rsIDs = []
for i in range(len(df_puro)):

    fila = df_puro.iloc[i]
        
    reg_sana = extrae_region_sana(fila)
    secuencias.append(str(reg_sana))
    etiquetas.append("Sano")
    # rsIDs.append(fila["SNPs_symbol"])
    
    reg_park = extrae_region_parkinson(fila, reg_sana)
    secuencias.append(str(reg_park))
    etiquetas.append("Riesgo_PD")
    # rsIDs.append(fila["SNPs_symbol"])

In [33]:
df_base_final = pd.DataFrame(list(zip(secuencias, etiquetas)), columns = ["Secuencia", "Etiqueta"])

In [34]:
df_base_final

,Secuencia,Etiqueta
0,atatactatgaatataactataatatacatatgtaaaatagctcag...,Sano
1,atatactatgaatataactataatatacatatgtaaaatagctcag...,Riesgo_PD
2,ccagggacatcatcaaaaggaatatccaggtgagtaggaagtgtgt...,Sano
3,ccagggacatcatcaaaaggaatatccaggtgagtaggaagtgtgt...,Riesgo_PD
4,atgttatcatgtaaatatctagttattgatcaataacgatgaccat...,Sano
...,...,...
1525,caccagccaccctgatcagtcagcagccatcaacatccatgcaaga...,Riesgo_PD
1526,agtctcttccattccttgaaggcacagagaggtgaagaagttgcag...,Sano
1527,agtctcttccattccttgaaggcacagagaggtgaagaagttgcag...,Riesgo_PD
1528,ttttttactagtcttagaaacagtgttggtgttttgtttttgtttt...,Sano


In [43]:
df_base_final.to_csv('datosGene4PD/base_flanqueantes_1530.csv', index = False)

In [28]:
def extrae_region_sana_OR(df_final_fila, ventana = 500):

    cromosoma = str(df_final_fila["Chr_corr"])
    pos_snp = df_final_fila["SNP_position_corr"]

    if float(df_final_fila["joint_phase_OR"]) <= 1:
        alelo_sano = df_final_fila["effect_allele"]

    else:
        alelo_sano = df_final_fila["alternate_allele"]

    secuencia_chr = genoma[cromosoma].seq

    id_snp_python = pos_snp - 1

    inicio = max(0, id_snp_python - ventana)
    fin = id_snp_python + ventana + 1

    molde_forward = str(secuencia_chr[inicio : fin]).upper()

    pos_relativa_snp = id_snp_python - inicio

    reg_sana_forward = molde_forward[:pos_relativa_snp] + alelo_sano + molde_forward[pos_relativa_snp + 1:]
    
    
    cadena = df_final_fila.get("Cadena")
    if cadena == 1 or cadena == "+":
        
        reg_sana = reg_sana_forward

    else:

        reg_sana = str(Seq(reg_sana_forward).reverse_complement())

    return reg_sana.lower()

In [29]:
def extrae_region_parkinson_OR(df_final_fila, reg_sana, ventana = 500):
    
    cadena = df_final_fila.get("Cadena")
    pos_snp = df_final_fila["SNP_position_corr"]

    if float(df_final_fila["joint_phase_OR"]) <= 1:

        alelo_parkinson = df_final_fila["alternate_allele"]

    else:
        
        alelo_parkinson = df_final_fila["effect_allele"]
        

    id_snp_python = pos_snp - 1
    inicio = max(0, id_snp_python - ventana)
    pos_relativa_snp = id_snp_python - inicio

    if cadena == 1 or cadena == "+":

        snp = pos_relativa_snp
        nuc = alelo_parkinson
    
    else:

        snp = len(reg_sana) - 1 - pos_relativa_snp
        nuc = str(Seq(alelo_parkinson).complement())

    nuc = nuc.lower()

    reg_park = reg_sana[:snp] + nuc + reg_sana[snp + 1:]

    return reg_park

In [64]:
secuencias = []
etiquetas = []
# rsIDs = []
for i in range(len(df_puro_OR)):

    fila = df_puro_OR.iloc[i]
        
    reg_sana = extrae_region_sana_OR(fila)
    secuencias.append(str(reg_sana))
    etiquetas.append("Sano")
    # rsIDs.append(fila["SNPs_symbol"])
    
    reg_park = extrae_region_parkinson_OR(fila, reg_sana)
    secuencias.append(str(reg_park))
    etiquetas.append("Riesgo_PD")
    # rsIDs.append(fila["SNPs_symbol"])

In [65]:
df_base_final_or = pd.DataFrame(list(zip(secuencias, etiquetas)), columns = ["Secuencia", "Etiqueta"])

In [66]:
df_base_final_or

,Secuencia,Etiqueta
0,atatactatgaatataactataatatacatatgtaaaatagctcag...,Sano
1,atatactatgaatataactataatatacatatgtaaaatagctcag...,Riesgo_PD
2,atgttatcatgtaaatatctagttattgatcaataacgatgaccat...,Sano
3,atgttatcatgtaaatatctagttattgatcaataacgatgaccat...,Riesgo_PD
4,ctttgaccaaccaactacttcaagttggagtttttacaacaccttt...,Sano
...,...,...
1413,caccagccaccctgatcagtcagcagccatcaacatccatgcaaga...,Riesgo_PD
1414,agtctcttccattccttgaaggcacagagaggtgaagaagttgcag...,Sano
1415,agtctcttccattccttgaaggcacagagaggtgaagaagttgcag...,Riesgo_PD
1416,ttttttactagtcttagaaacagtgttggtgttttgtttttgtttt...,Sano


In [67]:
df_base_final_or.to_csv('datosGene4PD/base_flanqueantes_OR_1418.csv', index = False)

PRUEBO A REDUCIR TAMAÑO DE VENTANA A 20

In [31]:
secuencias = []
etiquetas = []
rsIDs = []
for i in range(len(df_puro_OR)):

    fila = df_puro_OR.iloc[i]
        
    reg_sana = extrae_region_sana_OR(fila, ventana = 20)
    secuencias.append(str(reg_sana))
    etiquetas.append("Sano")
    rsIDs.append(fila["SNPs_symbol"])
    
    reg_park = extrae_region_parkinson_OR(fila, reg_sana, ventana = 20)
    secuencias.append(str(reg_park))
    etiquetas.append("Riesgo_PD")
    rsIDs.append(fila["SNPs_symbol"])

    print(f"Seq {i} preparada")

Seq 0 preparada
Seq 1 preparada
Seq 2 preparada
Seq 3 preparada
Seq 4 preparada
Seq 5 preparada
Seq 6 preparada
Seq 7 preparada
Seq 8 preparada
Seq 9 preparada
Seq 10 preparada
Seq 11 preparada
Seq 12 preparada
Seq 13 preparada
Seq 14 preparada
Seq 15 preparada
Seq 16 preparada
Seq 17 preparada
Seq 18 preparada
Seq 19 preparada
Seq 20 preparada
Seq 21 preparada
Seq 22 preparada
Seq 23 preparada
Seq 24 preparada
Seq 25 preparada
Seq 26 preparada
Seq 27 preparada
Seq 28 preparada
Seq 29 preparada
Seq 30 preparada
Seq 31 preparada
Seq 32 preparada
Seq 33 preparada
Seq 34 preparada
Seq 35 preparada
Seq 36 preparada
Seq 37 preparada
Seq 38 preparada
Seq 39 preparada
Seq 40 preparada
Seq 41 preparada
Seq 42 preparada
Seq 43 preparada
Seq 44 preparada
Seq 45 preparada
Seq 46 preparada
Seq 47 preparada
Seq 48 preparada
Seq 49 preparada
Seq 50 preparada
Seq 51 preparada
Seq 52 preparada
Seq 53 preparada
Seq 54 preparada
Seq 55 preparada
Seq 56 preparada
Seq 57 preparada
Seq 58 preparada
Seq 59 

In [ ]:
df_base_final_or = pd.DataFrame(list(zip(secuencias, etiquetas)), columns = ["Secuencia", "Etiqueta"])

In [ ]:
df_base_final_or

In [32]:
df_base_final_or.iloc[0]["Secuencia"]

'attcatcttccactgtgctaggaaggagaatgttcagaaac'

In [33]:
df_base_final_or.iloc[1]["Secuencia"]

'attcatcttccactgtgctatgaaggagaatgttcagaaac'

In [35]:
df_base_final_or.to_csv('datosGene4PD/base_flanqueantes_20_OR_1418.csv', index = False)

In [32]:
df_base_final_or_rsid = pd.DataFrame(list(zip(secuencias, etiquetas, rsIDs)), columns = ["Secuencia", "Etiqueta", "rsID"])

In [33]:
df_base_final_or_rsid

,Secuencia,Etiqueta,rsID
0,attcatcttccactgtgctaggaaggagaatgttcagaaac,Sano,rs757765789
1,attcatcttccactgtgctatgaaggagaatgttcagaaac,Riesgo_PD,rs757765789
2,tgctaaccctttttattttcagtgaactaatgcaggaaaaa,Sano,rs1994090
3,tgctaaccctttttattttccgtgaactaatgcaggaaaaa,Riesgo_PD,rs1994090
4,accagtcatctgtcaacttaatagcatacaaaaagacatca,Sano,rs2708453
...,...,...,...
1413,tacatcatctttaatattttttcactatttattctgaatac,Riesgo_PD,rs1941685
1414,tgctgaatattttaagcctacaattgagacctactgctcag,Sano,rs77351827
1415,tgctgaatattttaagcctataattgagacctactgctcag,Riesgo_PD,rs77351827
1416,taggtttaaaaactgggcatgtctcagagggcctatttatt,Sano,rs2248244


In [34]:
df_base_final_or_rsid.to_csv('datosGene4PD/base_flanqueantes_20_OR_1418_con_rsid.csv', index = False)